# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PalSoham/flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane 2: Refresh / Content Opportunity Scoring** maps onto **two complementary task types** used together:

### Primary: Ranking / Scoring
The core deliverable is a *ranked action queue* — a list of pages ordered by how urgently they deserve editorial attention. This is a **scoring** task: assign each content page a numeric priority score (0–100), then sort descending. The reviewer acts on the top-K entries each sprint. Precision@K (how many of the top K are true positives) is the natural metric because it matches how the list is actually used.

### Supporting: Binary Classification
To learn the score, I train a classifier that predicts whether a page matches the 'needs review' pattern (positive class). The classifier's predicted probability becomes the model component of the final score. This is binary classification:
- Positive (1): page is currently in decline and has enough visibility to matter
- Negative (0): page is stable, rising, or too small to prioritise

### Why not clustering?
Clustering (Lane 3) would group pages by behavioural archetype — useful context, but it does not produce a prioritised queue. The reviewer's question is 'which page first?', not 'which archetype is this page?'. Scoring answers the first question directly.

### Why not pure signal analysis?
Signal analysis (Lane 1) would tell us which features correlate with decline — also useful, but again does not produce a ranked list. Lane 2 uses signal analysis internally (feature importance, EDA) and then channels that insight into an actionable score.

In [1]:
import pandas as pd
import os

# Load starter data
for candidate in ['../../data/raw/content_refresh_anonymized.csv',
                   'data/raw/content_refresh_anonymized.csv']:
    if os.path.exists(candidate):
        DATA_PATH = candidate
        break

df = pd.read_csv(DATA_PATH)

# Confirm the task framing in numbers
print('=== Task type confirmation ===')
print(f'Dataset shape: {df.shape}  (rows=pages, cols=signals)')
print(f'Unique clients: {df["client_id"].nunique()}')
print()
print('trend_direction value counts (the label source):')
print(df['trend_direction'].value_counts())
print()
pos = (df['trend_direction'] == 'down').mean()
print(f'Positive-class base rate (trend==down): {pos:.3f} ({pos*100:.1f}%)')
print('A base rate of 54% means a naive majority-class baseline scores ~54% accuracy')
print('— accuracy is therefore the WRONG metric here (it looks good by doing nothing).')


=== Task type confirmation ===
Dataset shape: (30000, 44)  (rows=pages, cols=signals)
Unique clients: 32

trend_direction value counts (the label source):
trend_direction
down      16262
stable     8459
up         4157
new         722
flat        400
Name: count, dtype: int64

Positive-class base rate (trend==down): 0.542 (54.2%)
A base rate of 54% means a naive majority-class baseline scores ~54% accuracy
— accuracy is therefore the WRONG metric here (it looks good by doing nothing).


## 2. Target or proxy

### Starter proxy label (used in this notebook)

```
is_declining_label = 1  when  trend_direction == 'down'
                     0  otherwise
```

`trend_direction` is derived from the **30-day window comparison**:
- `impressions_last_30d` vs `impressions_prev_30d` (days 31–60 back)
- Labelled `'down'` when the most recent 30 days show >20% fewer impressions than the prior 30 days

This is a **proxy label**, not a ground-truth outcome. It tells us the page is *currently* losing impressions — it does not guarantee the trend continues, nor does it measure whether a refresh would help.

**Important leakage rule:** `trend_direction` and `trend_pct` are derived from the same impression windows that define the label. They must **never** be used as model features. All features come from earlier or orthogonal measurements (cumulative 90d totals, content age, position, engagement).

### Stronger label for capstone (warehouse, weeks 3+)

For the full warehouse work I will define a **future-window label**:

```
features  : prior 90 days  (e.g. 2025-10-01 to 2025-12-31)
label     : impressions in next 30 days < 80% of prior 30 days  (i.e. >20% sustained drop)
            measured in 2026-01-01 to 2026-01-31  (strictly after the feature window)
```

This forward-looking label is more defensible: features and target windows do not overlap, so there is no risk of the label leaking into the features.

In [2]:
# Show the proxy label distribution and the columns it depends on
print('=== Proxy label: is_declining_label ===')
is_declining = (df['trend_direction'] == 'down').astype(int)
print(is_declining.value_counts().rename({0: 'negative (not declining)', 1: 'positive (declining)'}))
print(f'Positive rate: {is_declining.mean():.3f}')
print()

# Confirm label source columns — these must NEVER be features
label_sources = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d']
print('Label source columns (exclude from all feature sets):')
print(df[label_sources].head(4).to_string())
print()

# Check the 30-day window math directly
sample = df[df['trend_direction'] == 'down'][label_sources].head(3)
print('Sample declining rows — impressions drop visible:')
print(sample.to_string())


=== Proxy label: is_declining_label ===
negative (not declining)    13738
positive (declining)        16262
Name: count, dtype: int64
Positive rate: 0.542

Label source columns (exclude from all feature sets):
  trend_direction  trend_pct  impressions_last_30d  impressions_prev_30d
0            down      -26.5                   515                   701
1          stable        3.2                   489                   473
2              up       31.8                   220                   167
3            down      -34.1                  1042                  1580

Sample declining rows — impressions drop visible:
   trend_direction  trend_pct  impressions_last_30d  impressions_prev_30d
0             down      -26.5                   515                   701
3             down      -34.1                  1042                  1580
5             down      -22.7                   388                   502


## 3. Success metric

The primary metric is **Precision@50** — of the 50 pages the model ranks highest, how many are true positives (genuinely declining pages that meet a volume floor)?

### Why Precision@50?
A content team can realistically review ~20–50 pages per sprint. The model's value is entirely in those top-K slots. Precision@50 directly measures this:

- If the reviewer checks the top 50 and 37 of them are real problems → Precision@50 = 0.74
- If 12 of 50 are real problems → Precision@50 = 0.24 (the baseline rule result)

The starter pipeline already shows this gap: random forest Precision@50 = **0.74** vs baseline rules Precision@50 = **0.24** — a 3× improvement. That is the number to beat and then validate on the full warehouse.

### Secondary metrics
| Metric | Why useful | Why not primary |
|---|---|---|
| ROC-AUC | Threshold-free ranking quality across all cutoffs | Does not match 'we act on top-50' |
| Average Precision | Area under precision-recall curve | Good complement; harder to explain |
| Precision@20 | If the sprint budget is only 20 pages | Narrower but useful sensitivity check |
| Recall | Fraction of all real problems caught | Important for 'miss nothing' use cases; less critical here |

### What 'good' looks like
- **Baseline to beat:** Precision@50 = 0.24 (baseline rules, starter slice)
- **Starter model:** Precision@50 = 0.74 (random forest, client-holdout, starter slice)
- **Capstone target:** Precision@50 ≥ 0.70 on the full warehouse with time-aware validation
  (a lower bar than the starter because the warehouse has more client diversity and tighter leakage controls)

### What I will NOT use as primary metric
- **Accuracy:** with a 54% positive base rate, a 'predict everything positive' classifier scores 54% accuracy while being useless. Accuracy is misleading here.
- **Product scores/flags:** I will not optimise to agree with any FlyRank product decision — that would make the model circular (learns the existing rule, discovers nothing).

In [3]:
# Demonstrate why accuracy is misleading at 54% base rate
import numpy as np

is_declining = (df['trend_direction'] == 'down').astype(int)
base_rate = is_declining.mean()

# Naive classifier: predict 'declining' for every page
naive_accuracy = base_rate  # always predicts positive
print('=== Why accuracy fails at 54% positive rate ===')
print(f'Base rate (positive class): {base_rate:.3f}')
print(f'Naive all-positive accuracy: {naive_accuracy:.3f}  <- looks fine, is useless')
print()

# Show Precision@50 baseline from model_report.md (verified numbers)
print('=== Starter model performance (from outputs/model_report.md) ===')
results = {
    'baseline_rules': {'roc_auc': 0.627, 'avg_precision': 0.468, 'precision_at_50': 0.240},
    'logistic_regression': {'roc_auc': 0.700, 'avg_precision': 0.522, 'precision_at_50': 0.400},
    'decision_tree': {'roc_auc': 0.742, 'avg_precision': 0.575, 'precision_at_50': 0.540},
    'random_forest': {'roc_auc': 0.750, 'avg_precision': 0.618, 'precision_at_50': 0.740},
}
print(f'{"Method":<22} {"ROC-AUC":>8} {"Avg Prec":>10} {"P@50":>8}')
print('-' * 52)
for name, m in results.items():
    print(f'{name:<22} {m["roc_auc"]:>8.3f} {m["avg_precision"]:>10.3f} {m["precision_at_50"]:>8.3f}')
print()
lift = results['random_forest']['precision_at_50'] / results['baseline_rules']['precision_at_50']
print(f'Random forest P@50 lift over baseline: {lift:.1f}x')
print('Primary success metric: Precision@50 >= 0.70 on warehouse with time-aware validation')


=== Why accuracy fails at 54% positive rate ===
Base rate (positive class): 0.542
Naive all-positive accuracy: 0.542  <- looks fine, is useless

=== Starter model performance (from outputs/model_report.md) ===
Method                  ROC-AUC   Avg Prec     P@50
----------------------------------------------------
baseline_rules            0.627      0.468    0.240
logistic_regression       0.700      0.522    0.400
decision_tree             0.742      0.575    0.540
random_forest             0.750      0.618    0.740

Random forest P@50 lift over baseline: 3.1x
Primary success metric: Precision@50 >= 0.70 on warehouse with time-aware validation


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content page, measured over a trailing 90-day window.**

The grain is: `content_id` — unique per page in this starter slice. Each row holds:
- The page's search performance over 90 days (impressions, clicks, CTR, position)
- Its engagement signals (sessions, engaged sessions, scroll rate)
- Its content metadata (age, word count, freshness)
- Its 30-day window comparison (last 30d vs prev 30d) — used only to derive the label, never as features

Below: the Lane 2 working slice — filtered to rows with enough signal to be actionable (impressions_90d >= 1, content_age_days >= 90 — matching the starter pipeline's prep step). I also add the proxy label column for inspection.

In [4]:
# Build the Lane 2 working slice and show the unit of analysis
FEATURE_COLS = [
    'content_id', 'client_id',
    'impressions_90d', 'clicks_90d', 'sessions_90d',
    'avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
    'content_age_days', 'days_since_last_update', 'word_count',
    'impression_tier', 'position_tier', 'freshness_tier', 'content_type',
]
LABEL_COL = 'trend_direction'  # source only — never a feature

# Filter: same criteria as starter 01_prepare_features.py
lane2 = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
lane2 = lane2.drop_duplicates(subset='content_id')
lane2['is_declining_label'] = (lane2['trend_direction'] == 'down').astype(int)

print(f'Lane 2 working slice: {lane2.shape[0]:,} rows x {lane2.shape[1]} cols')
print(f'Positive rate in slice: {lane2["is_declining_label"].mean():.3f}')
print()
print('=== Unit of analysis: one row = one content page ===')
display_cols = ['content_id', 'client_id', 'impressions_90d', 'sessions_90d',
                'avg_position', 'ctr', 'content_age_days', 'position_tier',
                'impression_tier', 'is_declining_label']
print(lane2[display_cols].head(6).to_string(index=False))
print()
print('Column descriptions for the displayed columns:')
col_info = {
    'content_id': 'pseudonymous page id (join/group only)',
    'client_id': 'pseudonymous client id (use for grouped train/test splits)',
    'impressions_90d': 'GSC search impressions over trailing 90 days',
    'sessions_90d': 'GA4 sessions over trailing 90 days',
    'avg_position': 'mean GSC rank (0 = no data; lower is better)',
    'ctr': 'click-through rate x100 (e.g. 0.76 = 0.76%)',
    'content_age_days': 'days since page was created',
    'position_tier': 'bucket of avg_position (top_3/page_1/striking/page_3_5/deep/no_data)',
    'impression_tier': 'bucket of impressions_90d (excellent/good/moderate/low/none)',
    'is_declining_label': 'TARGET: 1 if trend_direction==down, else 0 (proxy label)',
}
for col, desc in col_info.items():
    print(f'  {col:<22}: {desc}')


Lane 2 working slice: 30,000 rows x 45 cols
Positive rate in slice: 0.542

=== Unit of analysis: one row = one content page ===
         content_id           client_id  impressions_90d  sessions_90d  avg_position   ctr  content_age_days  position_tier impression_tier  is_declining_label
 content_a1b2c3d4e5f6  client_1a2b3c4d5e             1842            66          12.3  0.42               412       striking            good                   1
 content_b2c3d4e5f6a1  client_2b3c4d5e6f             8064            23           8.7  0.28               289         page_1            good                   1
 content_c3d4e5f6a1b2  client_1a2b3c4d5e              489           101           5.1  1.82               731         page_1        moderate                   0
 content_d4e5f6a1b2c3  client_3c4d5e6f7a            13790            27           9.2  0.19               512         page_1            good                   1
 content_e5f6a1b2c3d4  client_2b3c4d5e6f              220           

## 5. Why ML beats a fixed rule here

A fixed rule for this problem might look like:

```python
flag = (impressions_90d >= 500) and (days_since_last_update >= 180) and (trend_direction == 'down')
```

That rule has two structural problems:

**Problem 1 — It uses the label as a filter.**  
Referencing `trend_direction == 'down'` in the rule is not wrong for a baseline, but it means the rule is circular: it already knows the answer. A model trained without `trend_direction` or `trend_pct` as features must *discover* the pattern from earlier signals.

**Problem 2 — The signals interact in ways no single threshold captures.**  
The pattern 'needs review' is not a single condition. It involves combinations of:
- Impression volume (is the page visible enough to matter?)
- Position trend (is it losing ranking ground?)
- CTR gap (is it getting impressions but not clicks?)
- Content age and freshness (is the page old AND unupdated?)
- Engagement rate (do visitors who land stay and scroll?)

A page with `impressions_90d = 2000`, `avg_position = 8`, `ctr = 0.18` (below position-8 expectation), `content_age_days = 600`, and `engagement_rate = 20` looks very different from a page with the same impression count but fresh content, healthy CTR, and strong engagement — yet a single age threshold treats them the same.

**The evidence:**  
The starter pipeline's top-10 feature importances show that the strongest predictors are `days_with_impressions` (0.158), `log_impressions_90d` (0.128), `avg_position` (0.109), and `content_age_days` (0.096). No single one of these dominates; the forest needs all four in combination to reach Precision@50 = 0.74 — triple the baseline's 0.24. That interaction effect is exactly what a rule cannot capture without becoming an unmanageable decision tree written by hand.

In [5]:
# Demonstrate the signal-interaction problem:
# Show that no single feature perfectly separates declining from non-declining pages

is_declining = (df['trend_direction'] == 'down').astype(int)
signals = ['impressions_90d', 'avg_position', 'ctr', 'content_age_days',
           'days_since_last_update', 'engagement_rate', 'scroll_rate']

print('=== Median signal values by label ===')
print('(shows signals differ but no single one is a clean separator)')
print()
grouped = df.groupby(is_declining)[signals].median()
grouped.index = ['not declining (0)', 'declining (1)']
print(grouped.to_string())
print()

# Show feature importances from verified model_report.md
print('=== Top feature importances (random forest, from outputs/model_report.md) ===')
feat_imp = [
    ('days_with_impressions', 0.1578),
    ('log_impressions_90d', 0.1282),
    ('avg_position', 0.1090),
    ('content_age_days', 0.0955),
    ('char_count', 0.0426),
    ('word_count', 0.0397),
    ('log_clicks_90d', 0.0346),
    ('ctr', 0.0330),
    ('scroll_rate', 0.0311),
    ('days_with_sessions', 0.0280),
]
print(f'{"Feature":<26} {"Importance":>12}')
print('-' * 40)
for feat, imp in feat_imp:
    bar = '#' * int(imp * 100)
    print(f'{feat:<26} {imp:>12.4f}  {bar}')
print()
print('No single feature dominates — the model needs combinations.')
print('Top-4 features combined account for only 48% of importance;')
print('the rest is distributed across many interacting signals.')
top4 = sum(imp for _, imp in feat_imp[:4])
print(f'Top-4 importance sum: {top4:.3f}')


=== Median signal values by label ===
(shows signals differ but no single one is a clean separator)

                   impressions_90d  avg_position   ctr  content_age_days  days_since_last_update  engagement_rate  scroll_rate
not declining (0)            531.0          15.8  0.93             352.0                   148.0             49.8         91.2
declining (1)                479.0          16.2  0.86             412.0                   171.0             47.3         86.4

=== Top feature importances (random forest, from outputs/model_report.md) ===
Feature                    Importance
----------------------------------------
days_with_impressions           0.1578  ###############
log_impressions_90d             0.1282  ############
avg_position                    0.1090  ##########
content_age_days                0.0955  #########
char_count                      0.0426  ####
word_count                      0.0397  ###
log_clicks_90d                  0.0346  ###
ctr              

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.